# 🏙️ 城市扩张遥感分析 v2.0 — Jupyter Notebook 教学版

## 基于 Landsat + GEE + NDBI/NDVI/MNDWI 的多城市对比分析

本 Notebook 带你逐步了解每个分析环节。

**首次使用**：
1. 注册 GEE（免费）：https://earthengine.google.com/signup/
2. 终端认证：`earthengine authenticate`
3. 安装依赖：`pip install -r requirements.txt`

---
## 📦 第 1 步：导入模块

新架构下，每个功能都是独立的模块。

In [ ]:
import sys
sys.path.insert(0, '..')

import ee
from src.config import CITIES, ANALYSIS_CONFIG
from src.gee_utils import init_gee, get_roi, get_landsat_image
from src.indices import calc_ndbi, calc_ndvi, calc_mndwi, calc_all_indices
from src.urban_extraction import (
    extract_built_up, extract_built_up_refined,
    calc_built_up_area, calc_direction_stats, process_all_years
)
from src.accuracy import evaluate_accuracy, print_accuracy_report
from src.stats import compute_statistics, create_comparison_table

print('[✓] 所有模块导入成功！')
print(f'[→] 配置了 {len(CITIES)} 个城市: {", ".join(CITIES.keys())}')

---
## 🌍 第 2 步：初始化 GEE

In [ ]:
if init_gee():
    print('[✓] 可以开始分析了！')
else:
    print('[!] 请先完成 GEE 认证')
    print('[→] 在终端运行: earthengine authenticate')
    # 或者在 Notebook 内：ee.Authenticate(); ee.Initialize()

---
## 📍 第 3 步：选择城市 & 创建研究区

**💡 换城市？改 `city_name` 就行！想加新城市？去 `src/config.py`。**

In [ ]:
city_name = '三明市'  # ← 改成你想分析的城市
city_config = CITIES[city_name]

roi = get_roi(city_config['lon'], city_config['lat'], city_config['buffer'])

print(f'城市: {city_name}')
print(f'中心: ({city_config["lon"]}, {city_config["lat"]})')
print(f'半径: {city_config["buffer"]}m')
print(f'说明: {city_config["description"]}')

---
## 🛰️ 第 4 步：获取单一年份 Landsat 影像

试试获取 2025 年的影像。注意：第一次运行较慢（GEE 云端计算）。

In [ ]:
year = 2025
image, sensor = get_landsat_image(year, roi)
print(f'传感器: Landsat {sensor}')
print(f'波段数: {image.bandNames().size().getInfo()}')
print(f'波段: {image.bandNames().getInfo()}')

---
## 🔬 第 5 步：计算遥感指数

一步计算三个指数：**NDBI**（建筑）、**NDVI**（植被）、**MNDWI**（水体）

In [ ]:
image = calc_all_indices(image)

# 查看各指数的统计值
for index_name in ['NDBI', 'NDVI', 'MNDWI']:
    stats = image.select(index_name).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    )
    mean_val = ee.Number(stats.get(index_name)).getInfo()
    print(f'{index_name} 均值: {mean_val:.4f}')
    print(f'  → {"建筑多" if index_name=="NDBI" and mean_val > -0.05 else "植被好" if index_name=="NDVI" and mean_val > 0.3 else ""}')

---
## 🏗️ 第 6 步：提取建成区 + 计算面积

两种方法对比：
- **简单法**：只用 NDBI 阈值
- **精确法**：综合 NDBI + NDVI + MNDWI

In [ ]:
# 简单法
built_up_simple = extract_built_up(image)
area_simple = calc_built_up_area(built_up_simple, roi)

# 精确法
built_up_refined = extract_built_up_refined(image)
area_refined = calc_built_up_area(built_up_refined, roi)

print(f'简单法 (仅NDBI) — 建成区面积: {area_simple} km²')
print(f'精确法 (三指数) — 建成区面积: {area_refined} km²')
print(f'差异: {abs(area_refined - area_simple):.1f} km² (精确法排除了水体+植被的NDBI误判)')

---
## 🎯 第 7 步：精度验证

双阈值交叉验证 — 用严格 NDBI 阈值（> 0.05）作为伪参考，评估分类可靠性。

In [ ]:
# 需要 built_up 波段在影像上
image = image.addBands(built_up_simple)

accuracy = evaluate_accuracy(image, roi)
if accuracy:
    print_accuracy_report(accuracy)

---
## 📊 第 8 步：方向分析

八方向扇形统计 —— 看城市往哪个方向扩张最多。

In [ ]:
direction = calc_direction_stats(
    built_up_simple, roi,
    city_config['lon'], city_config['lat'],
    city_config['buffer']
)

print('各方向建成区占比:')
for d, p in sorted(direction.items(), key=lambda x: -x[1]):
    bar = '█' * int(p * 100)
    print(f'  {d}: {bar} {p:.1%}')

---
## 📈 第 9 步：批量分析所有年份

一键遍历 2000-2025 全部年份。

In [ ]:
years = ANALYSIS_CONFIG['years']

stats_list, images_dict, builtup_dict = process_all_years(
    years, roi, get_landsat_image, use_refined=False
)

# 统计
stats_df = compute_statistics(stats_list)
print('\n' + stats_df.to_string(index=False))

# 总增长
areas = stats_df['建成区面积(km²)'].tolist()
growth = areas[-1] - areas[0]
print(f'\n{years[0]}-{years[-1]} 总增长: {growth:.1f} km² (+{growth/areas[0]*100:.1f}%)')

---
## 🗺️ 第 10 步：交互式地图

在 Notebook 中直接查看任意年份的建成区叠图。

In [ ]:
import geemap

view_year = 2025  # ← 改成你想看的年份

Map = geemap.Map()
Map.centerObject(roi, 11)

# 真彩色底图
Map.addLayer(
    images_dict[view_year],
    {'bands': [2, 1, 0], 'min': 8000, 'max': 18000, 'gamma': 1.4},
    f'{view_year}年 真彩色'
)

# 建成区叠加（红色）
Map.addLayer(
    builtup_dict[view_year].updateMask(builtup_dict[view_year]),
    {'palette': ['red'], 'opacity': 0.5},
    f'{view_year}年 建成区'
)

# NDVI 图层
Map.addLayer(
    images_dict[view_year].select('NDVI'),
    {'palette': ['brown', 'yellow', 'green'], 'min': -0.2, 'max': 0.8},
    f'{view_year}年 NDVI'
)

Map

---
## 🏁 第 11 步：一键运行全部（命令行版）

如果你不想在 Notebook 里一步步走，直接在终端运行：

```bash
python src/main.py
```

会自动分析所有配置的城市，生成全部地图、图表、CSV。

---

## 📚 学习路线

按这个顺序读源码，理解整个项目：

1. **config.py** — 看看有哪些参数可以调
2. **indices.py** — 理解 NDBI/NDVI/MNDWI 公式和用法
3. **gee_utils.py** — 理解 GEE 影像获取和云掩膜
4. **urban_extraction.py** — 理解建成区提取和面积计算
5. **accuracy.py** — 理解精度验证方法
6. **visualization.py** — 理解图表生成
7. **main.py** — 看整个流程怎么串起来